# AIC 2026 — Normalize Object JSON trên local `/content`

Notebook này **chỉ xử lý riêng bước normalize object JSON** của pipeline Textual KIS.

## Mục tiêu

Thay vì đọc hàng trăm nghìn file JSON nhỏ trực tiếp từ Google Drive trong vòng lặp normalize:

```text
Google Drive: objects/
        │
        ├── đọc một lần để tạo ZIP
        ▼
/content/aic-object-normalize/objects_raw.zip
        │
        ├── unzip một lần
        ▼
/content/aic-object-normalize/objects/
        │
        ├── normalize hoàn toàn bằng local I/O
        ▼
/content/aic-object-normalize/output/
        ├── objects_index.parquet
        ├── keyframe_object_summary.parquet
        └── normalization_report.json
        │
        └── copy các file lớn cuối cùng về Drive
        ▼
MyDrive/data/processed-objects/
```

## Quy tắc normalize giữ nguyên

Raw JSON:

- `detection_class_entities`
- `detection_boxes`
- `detection_scores`
- bbox dạng `[ymin, xmin, ymax, xmax]`

Output detailed index:

- `global_id`
- `video_id`
- `keyframe_id`
- `label`
- `confidence`
- `xmin_norm`, `ymin_norm`, `xmax_norm`, `ymax_norm`
- `area_norm`
- `center_x_norm`, `center_y_norm`

Output keyframe summary:

- `global_id`
- `video_id`
- `keyframe_id`
- `objects_json`
- `has_raw_object_json`

Mặc định giữ detection có `confidence >= 0.20`.

> Raw `objects/` **không bị sửa hoặc xóa**. Derived artifacts được ghi vào `processed-objects/`.

In [ ]:
# ============================================================
# 0. INSTALL — chỉ cài dependency cần cho preprocessing
# ============================================================

!pip -q install -U pyarrow tqdm

print("Dependencies ready.")

In [ ]:
# ============================================================
# 1. MOUNT DRIVE + CONFIG
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import json
import math
import shutil
import subprocess
import time
from collections import Counter

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from PIL import Image

# -----------------------------
# Google Drive
# -----------------------------
DRIVE_ROOT = Path("/content/drive/MyDrive/data")

RAW_OBJECTS_DRIVE = DRIVE_ROOT / "objects"
PROCESSED_OBJECTS_DRIVE = DRIVE_ROOT / "processed-objects"

MASTER_CSV_DRIVE = DRIVE_ROOT / "keyframe_master.csv"
MASTER_PARQUET_DRIVE = DRIVE_ROOT / "retrieval-artifacts" / "master_keyframes.parquet"

PROCESSED_OBJECTS_DRIVE.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Local Colab SSD
# -----------------------------
LOCAL_ROOT = Path("/content/aic-object-normalize")
LOCAL_ZIP = LOCAL_ROOT / "objects_raw.zip"
LOCAL_EXTRACT_ROOT = LOCAL_ROOT / "source"
LOCAL_OBJECTS = LOCAL_EXTRACT_ROOT / "objects"

LOCAL_OUTPUT = LOCAL_ROOT / "output"
LOCAL_SHARDS = LOCAL_OUTPUT / "shards"

LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)
LOCAL_SHARDS.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Normalize settings
# -----------------------------
BASE_OBJECT_MIN_CONF = 0.20

# Chia master thành các shard local để:
# - giới hạn RAM
# - có checkpoint ở mức parquet shard
# - merge theo streaming ở cuối
NUM_LOCAL_SHARDS = 16

# Buffer detail detections trước mỗi lần ghi row-group vào parquet
DETAIL_BATCH_ROWS = 100_000

# Khi True, nếu đã có ZIP cache trên Drive thì notebook chỉ copy 1 file lớn xuống /content.
# Notebook KHÔNG tự tạo cache trên Drive mặc định vì lần chạy đầu sẽ tốn thêm một lần upload ZIP lớn.
DRIVE_ZIP_CACHE = DRIVE_ROOT / "objects_raw.zip"
USE_EXISTING_DRIVE_ZIP_CACHE = True

# Nếu muốn giữ ZIP vừa tạo cho lần Colab sau:
CACHE_NEW_ZIP_BACK_TO_DRIVE = False

# Nếu True, sau khi tạo xong từng shard local sẽ backup shard lên Drive.
# Không bắt buộc; mặc định False để ưu tiên tốc độ.
BACKUP_SHARDS_TO_DRIVE = False
DRIVE_SHARDS = PROCESSED_OBJECTS_DRIVE / "shards"
DRIVE_SHARDS.mkdir(parents=True, exist_ok=True)

FINAL_DETAIL_LOCAL = LOCAL_OUTPUT / "objects_index.parquet"
FINAL_SUMMARY_LOCAL = LOCAL_OUTPUT / "keyframe_object_summary.parquet"
FINAL_REPORT_LOCAL = LOCAL_OUTPUT / "normalization_report.json"

FINAL_DETAIL_DRIVE = PROCESSED_OBJECTS_DRIVE / "objects_index.parquet"
FINAL_SUMMARY_DRIVE = PROCESSED_OBJECTS_DRIVE / "keyframe_object_summary.parquet"
FINAL_REPORT_DRIVE = PROCESSED_OBJECTS_DRIVE / "normalization_report.json"

assert RAW_OBJECTS_DRIVE.exists(), f"Missing: {RAW_OBJECTS_DRIVE}"

print("RAW_OBJECTS_DRIVE:", RAW_OBJECTS_DRIVE)
print("LOCAL_ROOT:", LOCAL_ROOT)
print("OUTPUT:", PROCESSED_OBJECTS_DRIVE)
print("MIN_CONF:", BASE_OBJECT_MIN_CONF)

## 2. Load master keyframe index

Notebook ưu tiên `MyDrive/data/keyframe_master.csv` vì đây là master đang được dùng chung giữa các session.

Nếu CSV không tồn tại, notebook fallback sang:

```text
MyDrive/data/retrieval-artifacts/master_keyframes.parquet
```

Khi đọc CSV:

- ép `video_id` và `keyframe_id` thành string;
- xóa cột `Unnamed:*` nếu có;
- giữ zero-padding của `keyframe_id` như `001`, `002`, ...

In [ ]:
# ============================================================
# 2. LOAD MASTER
# ============================================================

def load_master():
    if MASTER_CSV_DRIVE.exists():
        print("Loading:", MASTER_CSV_DRIVE)
        df = pd.read_csv(
            MASTER_CSV_DRIVE,
            dtype={
                "video_id": "string",
                "keyframe_id": "string",
            }
        )
        df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed:")]
    elif MASTER_PARQUET_DRIVE.exists():
        print("Loading:", MASTER_PARQUET_DRIVE)
        df = pd.read_parquet(MASTER_PARQUET_DRIVE)
        df["video_id"] = df["video_id"].astype("string")
        df["keyframe_id"] = df["keyframe_id"].astype("string")
    else:
        raise FileNotFoundError(
            "Không tìm thấy cả keyframe_master.csv lẫn master_keyframes.parquet"
        )

    required = {"global_id", "video_id", "keyframe_id"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Master missing columns: {missing}")

    df = df.sort_values("global_id").reset_index(drop=True)

    # Chuyển global_id sang int64 rõ ràng
    df["global_id"] = pd.to_numeric(df["global_id"], errors="raise").astype("int64")

    return df

master = load_master()

print(f"Master rows : {len(master):,}")
print(f"Videos      : {master['video_id'].nunique():,}")
print("Columns     :", list(master.columns))
display(master.head())

assert master["global_id"].is_unique, "global_id must be unique"
assert master[["video_id", "keyframe_id"]].duplicated().sum() == 0, \
    "Duplicated (video_id, keyframe_id) in master"

# Không ép global_id phải bắt đầu từ 0, nhưng kiểm tra thứ tự tăng
assert master["global_id"].is_monotonic_increasing

## 3. Đưa raw objects từ Drive xuống `/content`

Ưu tiên theo thứ tự:

1. Nếu `MyDrive/data/objects_raw.zip` đã tồn tại và `USE_EXISTING_DRIVE_ZIP_CACHE=True`:
   - copy **một file ZIP lớn** xuống `/content`.
2. Nếu chưa có cache:
   - đọc `MyDrive/data/objects/` **một lần**;
   - tạo ZIP trực tiếp tại `/content/aic-object-normalize/objects_raw.zip`.
3. Giải nén ZIP tại `/content`.

Như vậy vòng normalize sau đó không còn đọc 177k file JSON nhỏ qua Google Drive.

In [ ]:
# ============================================================
# 3. ZIP/COPY RAW OBJECTS TO LOCAL /content
# ============================================================

def human_size(num_bytes):
    num = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if num < 1024.0:
            return f"{num:.2f} {unit}"
        num /= 1024.0
    return f"{num:.2f} PB"


def run_checked(cmd, cwd=None):
    print("$", " ".join(map(str, cmd)))
    if cwd is not None:
        print("  cwd:", cwd)
    subprocess.run(cmd, check=True, cwd=str(cwd) if cwd is not None else None)


def prepare_local_objects(force_rebuild_zip=False, force_reextract=False):
    LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

    # ---------- ZIP ----------
    if force_rebuild_zip and LOCAL_ZIP.exists():
        LOCAL_ZIP.unlink()

    if not LOCAL_ZIP.exists():
        if USE_EXISTING_DRIVE_ZIP_CACHE and DRIVE_ZIP_CACHE.exists():
            print("Found Drive ZIP cache:", DRIVE_ZIP_CACHE)
            print("Copying one large ZIP to /content ...")
            shutil.copy2(DRIVE_ZIP_CACHE, LOCAL_ZIP)
        else:
            print("No reusable ZIP cache. Creating ZIP directly in /content ...")
            print("This is the only step that traverses all small JSON files on Drive.")

            # -q: quiet, -r: recursive
            # cwd=DRIVE_ROOT để archive có top-level folder "objects/"
            run_checked(
                [
                    "zip", "-q", "-r",
                    str(LOCAL_ZIP),
                    "objects"
                ],
                cwd=DRIVE_ROOT,
            )

            if CACHE_NEW_ZIP_BACK_TO_DRIVE:
                print("Caching ZIP back to Drive for future sessions ...")
                tmp = DRIVE_ZIP_CACHE.with_suffix(".zip.tmp")
                shutil.copy2(LOCAL_ZIP, tmp)
                os.replace(tmp, DRIVE_ZIP_CACHE)

    print("ZIP ready:", LOCAL_ZIP, human_size(LOCAL_ZIP.stat().st_size))

    # ---------- EXTRACT ----------
    if force_reextract and LOCAL_EXTRACT_ROOT.exists():
        shutil.rmtree(LOCAL_EXTRACT_ROOT)

    if not LOCAL_OBJECTS.exists():
        LOCAL_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
        print("Extracting ZIP to local SSD ...")
        run_checked(["unzip", "-q", str(LOCAL_ZIP), "-d", str(LOCAL_EXTRACT_ROOT)])

    assert LOCAL_OBJECTS.exists(), f"Expected extracted folder: {LOCAL_OBJECTS}"

    print("Local objects ready:", LOCAL_OBJECTS)


prepare_local_objects(
    force_rebuild_zip=False,
    force_reextract=False,
)

## 4. Sanity check mapping trước khi normalize full

Bước này kiểm tra:

- path local có đúng dạng `objects/<video_id>/<keyframe_id>.json`;
- tỷ lệ JSON tìm thấy trên một sample;
- schema ba arrays có khớp độ dài;
- bbox đang ở normalized coordinates hay pixel coordinates.

Nếu filename raw object không khớp `keyframe_id`, **không nên chạy full normalization** cho tới khi sửa mapping.

In [ ]:
# ============================================================
# 4. SANITY CHECK RAW OBJECT JSON
# ============================================================

def local_object_path(video_id, keyframe_id):
    return LOCAL_OBJECTS / str(video_id) / f"{str(keyframe_id)}.json"


def inspect_raw_sample(sample_size=1000, seed=42):
    if len(master) <= sample_size:
        sample = master
    else:
        sample = master.sample(sample_size, random_state=seed)

    found = 0
    missing_examples = []
    bad_examples = []
    coord_modes = Counter()
    detection_counts = []

    for r in tqdm(sample.itertuples(index=False), total=len(sample), desc="Sanity sample"):
        p = local_object_path(r.video_id, r.keyframe_id)

        if not p.exists():
            if len(missing_examples) < 20:
                missing_examples.append(str(p))
            continue

        found += 1

        try:
            with p.open("r", encoding="utf-8") as f:
                d = json.load(f)

            labels = d.get("detection_class_entities", [])
            boxes = d.get("detection_boxes", [])
            scores = d.get("detection_scores", [])

            if not (len(labels) == len(boxes) == len(scores)):
                raise ValueError(
                    f"array lengths differ: labels={len(labels)}, "
                    f"boxes={len(boxes)}, scores={len(scores)}"
                )

            detection_counts.append(len(labels))

            if boxes:
                max_coord = max(float(x) for b in boxes for x in b)
                coord_modes["pixel" if max_coord > 1.5 else "normalized"] += 1
            else:
                coord_modes["empty"] += 1

        except Exception as e:
            if len(bad_examples) < 20:
                bad_examples.append((str(p), repr(e)))

    print(f"Sample rows  : {len(sample):,}")
    print(f"JSON found   : {found:,} ({found / max(1, len(sample)):.2%})")
    print("Coord modes  :", dict(coord_modes))

    if detection_counts:
        s = pd.Series(detection_counts)
        print("Detections/file:")
        display(s.describe().to_frame("count"))

    if missing_examples:
        print("\nMissing examples:")
        for x in missing_examples[:10]:
            print(" -", x)

    if bad_examples:
        print("\nBad JSON examples:")
        for p, e in bad_examples[:10]:
            print(" -", p, e)

    return {
        "sample_rows": len(sample),
        "found": found,
        "coord_modes": dict(coord_modes),
        "missing_examples": missing_examples,
        "bad_examples": bad_examples,
    }


sanity = inspect_raw_sample(sample_size=min(1000, len(master)))

# Nếu sample gần như không match thì thường là sai mapping/zero-padding.
assert sanity["found"] > 0, "Không tìm thấy JSON nào theo mapping master."

## 5. Helper cho pixel bbox

Baseline thường lưu bbox normalized. Tuy nhiên code vẫn hỗ trợ trường hợp bbox là pixel.

Thứ tự tìm `width/height`:

1. metadata trong chính JSON nếu có;
2. cột width/height trong master nếu có;
3. fallback đọc kích thước ảnh từ `image_path`.

Fallback (3) có thể chạm Google Drive nếu `image_path` nằm trên Drive, nhưng chỉ xảy ra khi raw bbox thực sự là pixel coordinates.

In [ ]:
# ============================================================
# 5. IMAGE SIZE RESOLUTION FOR PIXEL BBOX
# ============================================================

WIDTH_KEYS = ("image_width", "width", "img_width", "frame_width")
HEIGHT_KEYS = ("image_height", "height", "img_height", "frame_height")


def _first_positive_number(mapping, keys):
    for k in keys:
        if k in mapping and mapping[k] is not None:
            try:
                v = float(mapping[k])
                if v > 0:
                    return v
            except Exception:
                pass
    return None


def resolve_image_size(raw_json, master_row):
    # 1) raw JSON
    W = _first_positive_number(raw_json, WIDTH_KEYS)
    H = _first_positive_number(raw_json, HEIGHT_KEYS)
    if W and H:
        return float(W), float(H), "json"

    # 2) master columns
    row_dict = master_row._asdict() if hasattr(master_row, "_asdict") else dict(master_row)

    W = _first_positive_number(row_dict, WIDTH_KEYS)
    H = _first_positive_number(row_dict, HEIGHT_KEYS)
    if W and H:
        return float(W), float(H), "master"

    # 3) image_path fallback
    image_path = row_dict.get("image_path")
    if image_path:
        p = Path(str(image_path))
        if p.exists():
            with Image.open(p) as im:
                return float(im.width), float(im.height), "image_path"

    raise ValueError(
        "Pixel bbox detected but image width/height cannot be resolved. "
        f"video_id={row_dict.get('video_id')}, keyframe_id={row_dict.get('keyframe_id')}"
    )

## 6. Normalize theo local shards

Mỗi shard ghi riêng:

```text
/content/aic-object-normalize/output/shards/
├── objects_index.part-00-of-16.parquet
├── keyframe_object_summary.part-00-of-16.parquet
├── stats.part-00-of-16.json
├── ...
```

Nếu rerun cùng runtime, shard đã có đủ parquet + stats sẽ được skip.

### Summary logic

Với mỗi label sau threshold:

```json
{
  "person": {
    "count": 2,
    "max_conf": 0.94,
    "max_area": 0.31
  }
}
```

- label trong detailed index giữ `str(label).strip()`;
- key trong summary dùng lowercase;
- bbox được clamp về `[0, 1]`;
- `area_norm = width_norm * height_norm`;
- center được tính từ bbox normalized.

In [ ]:
# ============================================================
# 6. NORMALIZE LOCAL SHARDS
# ============================================================

DETAIL_SCHEMA = pa.schema([
    ("global_id", pa.int64()),
    ("video_id", pa.string()),
    ("keyframe_id", pa.string()),
    ("label", pa.string()),
    ("confidence", pa.float32()),
    ("xmin_norm", pa.float32()),
    ("ymin_norm", pa.float32()),
    ("xmax_norm", pa.float32()),
    ("ymax_norm", pa.float32()),
    ("area_norm", pa.float32()),
    ("center_x_norm", pa.float32()),
    ("center_y_norm", pa.float32()),
])

SUMMARY_SCHEMA = pa.schema([
    ("global_id", pa.int64()),
    ("video_id", pa.string()),
    ("keyframe_id", pa.string()),
    ("objects_json", pa.string()),
    ("has_raw_object_json", pa.bool_()),
])


def shard_paths(shard_id, num_shards=NUM_LOCAL_SHARDS):
    suffix = f"part-{shard_id:02d}-of-{num_shards:02d}"
    return {
        "detail": LOCAL_SHARDS / f"objects_index.{suffix}.parquet",
        "summary": LOCAL_SHARDS / f"keyframe_object_summary.{suffix}.parquet",
        "stats": LOCAL_SHARDS / f"stats.{suffix}.json",
    }


def drive_shard_paths(shard_id, num_shards=NUM_LOCAL_SHARDS):
    suffix = f"part-{shard_id:02d}-of-{num_shards:02d}"
    return {
        "detail": DRIVE_SHARDS / f"objects_index.{suffix}.parquet",
        "summary": DRIVE_SHARDS / f"keyframe_object_summary.{suffix}.parquet",
        "stats": DRIVE_SHARDS / f"stats.{suffix}.json",
    }


def flush_detail_buffer(writer, buffer):
    if not buffer:
        return writer, []

    table = pa.Table.from_pylist(buffer, schema=DETAIL_SCHEMA)

    if writer is None:
        raise RuntimeError("Detail writer should already be initialized.")

    writer.write_table(table)
    return writer, []


def normalize_one_shard(
    shard_id,
    num_shards=NUM_LOCAL_SHARDS,
    min_conf=BASE_OBJECT_MIN_CONF,
    detail_batch_rows=DETAIL_BATCH_ROWS,
    force=False,
):
    paths = shard_paths(shard_id, num_shards)

    complete = all(paths[k].exists() for k in ("detail", "summary", "stats"))
    if complete and not force:
        print(f"[SKIP] shard {shard_id:02d}/{num_shards:02d} already complete")
        return json.loads(paths["stats"].read_text(encoding="utf-8"))

    # Clean partial outputs
    for p in paths.values():
        if p.exists():
            p.unlink()

    # Contiguous split keeps final merge naturally ordered by global_id.
    index_chunks = np.array_split(np.arange(len(master)), num_shards)
    idxs = index_chunks[shard_id]
    shard_df = master.iloc[idxs]

    detail_writer = pq.ParquetWriter(
        paths["detail"],
        DETAIL_SCHEMA,
        compression="zstd",
    )

    detail_buffer = []
    summary_rows = []

    stats = {
        "shard_id": int(shard_id),
        "num_shards": int(num_shards),
        "master_rows": int(len(shard_df)),
        "found": 0,
        "missing": 0,
        "bad": 0,
        "detections_raw": 0,
        "detections_kept": 0,
        "pixel_json_files": 0,
        "normalized_json_files": 0,
        "empty_json_files": 0,
        "image_size_sources": {},
        "bad_examples": [],
    }

    image_size_sources = Counter()

    try:
        iterator = shard_df.itertuples(index=False)

        for r in tqdm(
            iterator,
            total=len(shard_df),
            desc=f"Normalize shard {shard_id:02d}",
        ):
            obj_path = local_object_path(r.video_id, r.keyframe_id)

            if not obj_path.exists():
                stats["missing"] += 1
                summary_rows.append({
                    "global_id": int(r.global_id),
                    "video_id": str(r.video_id),
                    "keyframe_id": str(r.keyframe_id),
                    "objects_json": "{}",
                    "has_raw_object_json": False,
                })
                continue

            stats["found"] += 1

            try:
                with obj_path.open("r", encoding="utf-8") as f:
                    d = json.load(f)

                labels = d.get("detection_class_entities", [])
                boxes = d.get("detection_boxes", [])
                scores = d.get("detection_scores", [])

                if not (len(labels) == len(boxes) == len(scores)):
                    raise ValueError(
                        f"array lengths differ: labels={len(labels)}, "
                        f"boxes={len(boxes)}, scores={len(scores)}"
                    )

                stats["detections_raw"] += len(labels)

                pixel_mode = False
                if boxes:
                    max_coord = max(float(x) for b in boxes for x in b)
                    pixel_mode = max_coord > 1.5

                if not boxes:
                    stats["empty_json_files"] += 1
                elif pixel_mode:
                    stats["pixel_json_files"] += 1
                else:
                    stats["normalized_json_files"] += 1

                W = H = None
                if pixel_mode:
                    W, H, size_source = resolve_image_size(d, r)
                    image_size_sources[size_source] += 1

                summary = {}

                for label, box, score in zip(labels, boxes, scores):
                    score = float(score)

                    if not math.isfinite(score):
                        continue

                    if score < min_conf:
                        continue

                    if len(box) != 4:
                        raise ValueError(f"bbox does not have 4 values: {box}")

                    ymin, xmin, ymax, xmax = map(float, box)

                    if not all(math.isfinite(x) for x in (ymin, xmin, ymax, xmax)):
                        continue

                    if pixel_mode:
                        xmin, xmax = xmin / W, xmax / W
                        ymin, ymax = ymin / H, ymax / H

                    xmin, ymin, xmax, ymax = [
                        max(0.0, min(1.0, x))
                        for x in (xmin, ymin, xmax, ymax)
                    ]

                    w = max(0.0, xmax - xmin)
                    h = max(0.0, ymax - ymin)
                    area = w * h

                    label_s = str(label).strip()
                    if not label_s:
                        continue

                    detail_buffer.append({
                        "global_id": int(r.global_id),
                        "video_id": str(r.video_id),
                        "keyframe_id": str(r.keyframe_id),
                        "label": label_s,
                        "confidence": np.float32(score),
                        "xmin_norm": np.float32(xmin),
                        "ymin_norm": np.float32(ymin),
                        "xmax_norm": np.float32(xmax),
                        "ymax_norm": np.float32(ymax),
                        "area_norm": np.float32(area),
                        "center_x_norm": np.float32((xmin + xmax) / 2.0),
                        "center_y_norm": np.float32((ymin + ymax) / 2.0),
                    })

                    stats["detections_kept"] += 1

                    item = summary.setdefault(
                        label_s.lower(),
                        {
                            "count": 0,
                            "max_conf": 0.0,
                            "max_area": 0.0,
                        },
                    )
                    item["count"] += 1
                    item["max_conf"] = max(item["max_conf"], score)
                    item["max_area"] = max(item["max_area"], area)

                    if len(detail_buffer) >= detail_batch_rows:
                        detail_writer, detail_buffer = flush_detail_buffer(
                            detail_writer,
                            detail_buffer,
                        )

                summary_rows.append({
                    "global_id": int(r.global_id),
                    "video_id": str(r.video_id),
                    "keyframe_id": str(r.keyframe_id),
                    "objects_json": json.dumps(
                        summary,
                        ensure_ascii=False,
                        separators=(",", ":"),
                    ),
                    "has_raw_object_json": True,
                })

            except Exception as e:
                stats["bad"] += 1

                if len(stats["bad_examples"]) < 20:
                    stats["bad_examples"].append({
                        "path": str(obj_path),
                        "error": repr(e),
                    })

                # Giữ semantics giống notebook gốc:
                # raw file tồn tại nhưng parse/normalize lỗi -> has_raw_object_json=True
                summary_rows.append({
                    "global_id": int(r.global_id),
                    "video_id": str(r.video_id),
                    "keyframe_id": str(r.keyframe_id),
                    "objects_json": "{}",
                    "has_raw_object_json": True,
                })

        # Flush detail cuối
        detail_writer, detail_buffer = flush_detail_buffer(
            detail_writer,
            detail_buffer,
        )

    finally:
        detail_writer.close()

    # Summary luôn có đúng 1 row / master keyframe trong shard
    summary_table = pa.Table.from_pylist(summary_rows, schema=SUMMARY_SCHEMA)
    pq.write_table(
        summary_table,
        paths["summary"],
        compression="zstd",
    )

    stats["image_size_sources"] = dict(image_size_sources)
    paths["stats"].write_text(
        json.dumps(stats, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    if BACKUP_SHARDS_TO_DRIVE:
        dpaths = drive_shard_paths(shard_id, num_shards)
        for key in ("detail", "summary", "stats"):
            shutil.copy2(paths[key], dpaths[key])

    print(
        f"[DONE] shard {shard_id:02d}: "
        f"rows={stats['master_rows']:,}, "
        f"found={stats['found']:,}, "
        f"missing={stats['missing']:,}, "
        f"bad={stats['bad']:,}, "
        f"kept={stats['detections_kept']:,}"
    )

    return stats

In [ ]:
# ============================================================
# 7. RUN ALL LOCAL SHARDS
# ============================================================

all_stats = []

for shard_id in range(NUM_LOCAL_SHARDS):
    stats = normalize_one_shard(
        shard_id=shard_id,
        num_shards=NUM_LOCAL_SHARDS,
        min_conf=BASE_OBJECT_MIN_CONF,
        detail_batch_rows=DETAIL_BATCH_ROWS,
        force=False,
    )
    all_stats.append(stats)

print("\nAll shards completed.")

## 8. Merge shards bằng streaming

Không dùng `pd.concat()` trên toàn bộ detailed index để tránh giữ toàn bộ object detections trong RAM.

Mỗi parquet shard được đọc theo record batch và ghi tuần tự vào file cuối.

In [ ]:
# ============================================================
# 8. STREAMING MERGE
# ============================================================

def merge_parquet_streaming(input_paths, output_path, schema, batch_size=250_000):
    if output_path.exists():
        output_path.unlink()

    writer = pq.ParquetWriter(
        output_path,
        schema,
        compression="zstd",
    )

    total_rows = 0

    try:
        for p in tqdm(input_paths, desc=f"Merge {output_path.name}"):
            pf = pq.ParquetFile(p)

            for batch in pf.iter_batches(batch_size=batch_size):
                table = pa.Table.from_batches([batch], schema=schema)
                writer.write_table(table)
                total_rows += table.num_rows
    finally:
        writer.close()

    print(f"Merged {total_rows:,} rows -> {output_path}")
    return total_rows


detail_shards = [
    shard_paths(i)["detail"]
    for i in range(NUM_LOCAL_SHARDS)
]
summary_shards = [
    shard_paths(i)["summary"]
    for i in range(NUM_LOCAL_SHARDS)
]

assert all(p.exists() for p in detail_shards)
assert all(p.exists() for p in summary_shards)

detail_rows = merge_parquet_streaming(
    detail_shards,
    FINAL_DETAIL_LOCAL,
    DETAIL_SCHEMA,
)

summary_rows = merge_parquet_streaming(
    summary_shards,
    FINAL_SUMMARY_LOCAL,
    SUMMARY_SCHEMA,
)

assert summary_rows == len(master), (
    f"Summary row count mismatch: summary={summary_rows}, master={len(master)}"
)

## 9. Validation output

Các kiểm tra cuối:

- summary có đúng 1 row / keyframe;
- `global_id` summary unique;
- confidence không thấp hơn threshold;
- bbox nằm trong `[0, 1]`;
- area không âm;
- thống kê found / missing / bad;
- schema output tương thích với notebook retrieval chính.

In [ ]:
# ============================================================
# 9. VALIDATE FINAL OUTPUT
# ============================================================

summary_df = pd.read_parquet(FINAL_SUMMARY_LOCAL)

assert len(summary_df) == len(master)
assert summary_df["global_id"].is_unique
assert summary_df["global_id"].is_monotonic_increasing

print("Summary:")
display(summary_df.head())
print(summary_df["has_raw_object_json"].value_counts(dropna=False))

# Đọc sample detailed index thay vì load toàn bộ vào RAM.
detail_pf = pq.ParquetFile(FINAL_DETAIL_LOCAL)
detail_total_rows = detail_pf.metadata.num_rows

print(f"\nDetailed rows: {detail_total_rows:,}")
print("Detailed schema:")
print(detail_pf.schema_arrow)

sample_batches = []
sample_rows_target = min(100_000, detail_total_rows)

for batch in detail_pf.iter_batches(batch_size=sample_rows_target):
    sample_batches.append(pa.Table.from_batches([batch]))
    break

if sample_batches:
    detail_sample = sample_batches[0].to_pandas()

    assert (detail_sample["confidence"] >= BASE_OBJECT_MIN_CONF - 1e-6).all()

    bbox_cols = [
        "xmin_norm",
        "ymin_norm",
        "xmax_norm",
        "ymax_norm",
        "center_x_norm",
        "center_y_norm",
    ]

    for c in bbox_cols:
        assert detail_sample[c].between(0.0, 1.0).all(), f"Out-of-range: {c}"

    assert (detail_sample["area_norm"] >= 0.0).all()
    assert (detail_sample["area_norm"] <= 1.0 + 1e-6).all()

    print("\nDetailed sample:")
    display(detail_sample.head())

    print("\nTop labels in sample:")
    display(
        detail_sample["label"]
        .value_counts()
        .head(20)
        .to_frame("count")
    )

# Aggregate shard stats
report = {
    "master_rows": int(len(master)),
    "videos": int(master["video_id"].nunique()),
    "min_conf": float(BASE_OBJECT_MIN_CONF),
    "num_local_shards": int(NUM_LOCAL_SHARDS),
    "detail_rows": int(detail_total_rows),
    "summary_rows": int(len(summary_df)),
    "found": int(sum(s["found"] for s in all_stats)),
    "missing": int(sum(s["missing"] for s in all_stats)),
    "bad": int(sum(s["bad"] for s in all_stats)),
    "detections_raw": int(sum(s["detections_raw"] for s in all_stats)),
    "detections_kept": int(sum(s["detections_kept"] for s in all_stats)),
    "pixel_json_files": int(sum(s["pixel_json_files"] for s in all_stats)),
    "normalized_json_files": int(sum(s["normalized_json_files"] for s in all_stats)),
    "empty_json_files": int(sum(s["empty_json_files"] for s in all_stats)),
    "detail_file_bytes": int(FINAL_DETAIL_LOCAL.stat().st_size),
    "summary_file_bytes": int(FINAL_SUMMARY_LOCAL.stat().st_size),
}

FINAL_REPORT_LOCAL.write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("\nNormalization report:")
print(json.dumps(report, ensure_ascii=False, indent=2))

## 10. Copy final artifacts về Drive

Chỉ có vài file lớn được ghi lên Drive:

```text
MyDrive/data/processed-objects/
├── objects_index.parquet
├── keyframe_object_summary.parquet
└── normalization_report.json
```

Raw `MyDrive/data/objects/` không bị thay đổi.

In [ ]:
# ============================================================
# 10. COPY FINAL OUTPUTS BACK TO GOOGLE DRIVE
# ============================================================

def atomic_copy_to_drive(src, dst):
    src = Path(src)
    dst = Path(dst)

    dst.parent.mkdir(parents=True, exist_ok=True)

    tmp = dst.parent / f".{dst.name}.tmp"

    if tmp.exists():
        tmp.unlink()

    print(f"Copying {src.name}: {human_size(src.stat().st_size)}")
    shutil.copy2(src, tmp)
    os.replace(tmp, dst)

    assert dst.exists()
    assert dst.stat().st_size == src.stat().st_size

    print("Saved:", dst)


atomic_copy_to_drive(FINAL_DETAIL_LOCAL, FINAL_DETAIL_DRIVE)
atomic_copy_to_drive(FINAL_SUMMARY_LOCAL, FINAL_SUMMARY_DRIVE)
atomic_copy_to_drive(FINAL_REPORT_LOCAL, FINAL_REPORT_DRIVE)

print("\nFinal artifacts are safely stored in:")
print(PROCESSED_OBJECTS_DRIVE)

## 11. Final verification từ Drive

Đọc lại **chỉ parquet output lớn** từ Drive để xác nhận file copy thành công. Đây không còn là pattern hàng trăm nghìn small-file reads.

In [ ]:
# ============================================================
# 11. FINAL VERIFY
# ============================================================

drive_summary = pd.read_parquet(FINAL_SUMMARY_DRIVE)

assert len(drive_summary) == len(master)
assert drive_summary["global_id"].is_unique

drive_detail_pf = pq.ParquetFile(FINAL_DETAIL_DRIVE)

print("Drive summary rows :", f"{len(drive_summary):,}")
print("Drive detail rows  :", f"{drive_detail_pf.metadata.num_rows:,}")
print("Drive detail file  :", human_size(FINAL_DETAIL_DRIVE.stat().st_size))
print("Drive summary file :", human_size(FINAL_SUMMARY_DRIVE.stat().st_size))

display(drive_summary.head())

print("\nNORMALIZE OBJECTS: COMPLETE")

## 12. Optional cleanup `/content`

Chỉ chạy cell này **sau khi** đã verify output trên Drive.

- Xóa extracted JSON local để giải phóng SSD.
- Có thể giữ `objects_raw.zip` nếu còn muốn rerun trong cùng Colab runtime.

In [ ]:
# ============================================================
# 12. OPTIONAL LOCAL CLEANUP
# ============================================================

CLEAN_EXTRACTED_OBJECTS = False
DELETE_LOCAL_ZIP = False
DELETE_LOCAL_SHARDS = False

if CLEAN_EXTRACTED_OBJECTS and LOCAL_EXTRACT_ROOT.exists():
    shutil.rmtree(LOCAL_EXTRACT_ROOT)
    print("Removed:", LOCAL_EXTRACT_ROOT)

if DELETE_LOCAL_ZIP and LOCAL_ZIP.exists():
    LOCAL_ZIP.unlink()
    print("Removed:", LOCAL_ZIP)

if DELETE_LOCAL_SHARDS and LOCAL_SHARDS.exists():
    shutil.rmtree(LOCAL_SHARDS)
    print("Removed:", LOCAL_SHARDS)

print("Cleanup configuration applied.")